# Module 3 POC Training -- EfficientNet-B4 + Weibull Survival Head

Trains Module 3 (time-to-progression estimation) on Tianjin's baseline -> 2-year-follow-up
DR grades, resolved per eye (Organized_Data of Patients.xlsx grades DR per eye, not per
patient): an EfficientNet-B4 backbone (ImageNet-pretrained, independently trained from
Module 1's classifier -- no shared weights) with a Weibull survival head, using the LBS
stratum (low/medium/high, see `module1/compute_lbs.py`) as an auxiliary input.

**Run this after notebook 05** (it reuses Tianjin's `module1_outputs_tianjin.pt` cache and
`laterality_resolved.csv` -- Module 3 has no placeholder fallback for either).

**Round 4:** training keeps every eye (baseline grade is an input), but the metrics are ALSO reported on
the in-scope eyes (baseline >= mild NPDR), with 95 % patient-bootstrap intervals and two non-image
baselines on the same patients (train event rate per baseline grade; logistic regression on baseline
grade + log LBS + Module 1's grade) -- the image model only adds something if it beats those. More of the
EfficientNet-B4 backbone stays frozen (`FREEZE_BLOCKS = 8`: only the head conv + survival head train),
since round 3 over-fitted (train NLL 0.80 -> 0.07, val flat, test AUROC 0.55).

Tianjin's fixed 2-year follow-up window means this can only support a "probability of
progression by 2 years" claim, not a full time-to-event curve at arbitrary horizons. Report
it as a population-level probabilistic estimate with the AUROC below, not a per-patient point
prediction.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('⚠ Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'R5-Round3Baseline-Fixes'  # <-- round 5; 'AllModuleEdits-PreFinals' = rounds 3/4, 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"

# Which Module 1 models the pipeline uses -- set the SAME value in notebooks 04_05 and 06-10:
#   'drgnet' -- notebook 11's DRG-Net replication checkpoints (module1_runs/drgnet_replication/)
#   'poc'    -- notebooks 02/03's original POC checkpoints (module1_runs/)
# Everything built from Module 1's outputs (Module 1 caches, Module 2 runs, figures) gets a
# '_drgnet' suffix for 'drgnet', so the two pipelines never overwrite each other's results.
MODULE1_SOURCE = 'drgnet_combined'
assert MODULE1_SOURCE in ('drgnet_combined', 'drgnet', 'poc'), MODULE1_SOURCE
# New Drive folders for this branch's runs: every Module 1 cache / Module 2 / Module 3 folder below
# gets RUN_SUFFIX, so the mod-2-NonLongIntegration results stay untouched for before/after
# comparison. '' = read/write the earlier folders again.
RUN_SUFFIX = '_prefinals'
RUN_TAG = {'drgnet_combined': '_drgnet_comb', 'drgnet': '_drgnet', 'poc': ''}[MODULE1_SOURCE] + RUN_SUFFIX
# Round 4: every Module 1 cache / Module 2 / Module 3 folder ALSO gets ROUND_TAG, so round 3's results
# (no ROUND_TAG) stay untouched for before/after comparison -- the SAME value in notebooks 04_05 and 06-10.
# Model folders (notebook 11) and the laterality file keep RUN_SUFFIX only. '' = round 3's folders.
ROUND_TAG = '_r5'
# Round 5 (branch R5-Round3Baseline-Fixes): round 3's Module 2 settings (128 px, no grader loss, round 3's
# best-epoch rule) + the data fixes (text-safe retina crop, SIFT failures dropped). '_r4' = round 4's folders.
RUN_TAG += ROUND_TAG
# Image preprocessing for every script this notebook runs (fundus_preprocess.py), the same in 04_05-10:
#   DRP_RETINA_CROP '1' = crop to the retina and pad to square before resizing (Tianjin's 3:2 and LongDR's
#     2000x1312 photos were squashed into ovals); '0' = round 3's plain resize.
#   DRP_LONGDR_FOLDER 'FundusImagesColor' = LongDR's raw colour photos. Round 3 used FundusImagesNormalized,
#     on which the lesion segmenters (trained on raw photos) marked 10-30 % of a NORMAL retina as lesion
#     (LBS 0.1-0.34 at grade 0, vs ~0.001 on Tianjin/FIRE) -- 559 of the 917 Module 2 pairs.
os.environ['DRP_RETINA_CROP'] = '1'
os.environ['DRP_LONGDR_FOLDER'] = 'FundusImagesColor'
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', *(['drgnet_replication'] if MODULE1_SOURCE != 'poc' else []))
# Notebook 11 on this branch writes its models to drgnet_replication<RUN_SUFFIX>/ (tiled evaluation,
# 200 segmentation epochs, optional EyePACS grader); use those once they exist.
if MODULE1_SOURCE != 'poc' and os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'segmentation')):
    MODULE1_RUNS += RUN_SUFFIX
elif MODULE1_SOURCE != 'poc' and RUN_SUFFIX:
    print(f'\u26a0 {MODULE1_RUNS + RUN_SUFFIX} not found -- using the earlier Module 1 models in {MODULE1_RUNS}. '
          'Run notebook 11 on this branch first for the re-trained ones.')
# Lesion segmenters: <MODULE1_RUNS>/segmentation/<SEG_MODEL_PREFIX><lesion>/model_2.pth.tar --
# FGADR + DDR combined (notebook 11) or FGADR only. The classifier is FGADR's either way.
SEG_MODEL_PREFIX = 'comb_models_' if MODULE1_SOURCE == 'drgnet_combined' else 'models_FGADR_NO_TATL_'
# Round 4: notebook 11's fine-tuned combined segmenters (comb_ft_models_*) once all four exist.
if MODULE1_SOURCE == 'drgnet_combined':
    if all(os.path.isfile(os.path.join(MODULE1_RUNS, 'segmentation', f'comb_ft_models_{_l}', 'model_2.pth.tar'))
           for _l in ('ex', 'he', 'ma', 'se')):
        SEG_MODEL_PREFIX = 'comb_ft_models_'
    elif ROUND_TAG:
        print('\u26a0 Fine-tuned segmenters (comb_ft_models_*) not all on Drive yet -- using the 200-epoch '
              'comb_models_*. Run notebook 11 (RUN["comb_seg_ft"]) first for round 4.')
print('Lesion segmenters:', SEG_MODEL_PREFIX + '<lesion>')
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_TAG)
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
print('Module 1 source:', MODULE1_SOURCE, '| models:', MODULE1_RUNS, '| caches:', MODULE1_CACHE_DIR)
TIANJIN_CACHE = os.path.join(MODULE1_CACHE_DIR, 'module1_outputs_tianjin.pt')
assert os.path.isfile(TIANJIN_CACHE), (
    f"Not found: {TIANJIN_CACHE} -- run notebook 04_05 first (same MODULE1_SOURCE)."
)

Mounted at /content/drive
Google Drive: 153.6 GB free of 214.7 GB
GitHub Personal Access Token (repo scope): ··········
Lesion segmenters: comb_ft_models_<lesion>
Module 1 source: drgnet_combined | models: /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals | caches: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5


In [2]:
# Unzip Tianjin locally (same convention as notebooks 05/06).
import glob

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found -- see notebook 05 for troubleshooting.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)
print('Tianjin manifest found:', os.path.isfile(os.path.join(TIANJIN_DIR, 'corrected_manifest.csv')))

/content/data
Tianjin manifest found: True


In [3]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl scikit-learn opencv-python-headless

/content
Cloning into 'DRProgression'...
remote: Enumerating objects: 1114, done.
remote: Counting objects: 100% (280/280), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 1114 (delta 184), reused 198 (delta 143), pack-reused 834 (from 1)
Receiving objects: 100% (1114/1114), 42.25 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (669/669), done.
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc


In [4]:
# Laterality resolution: Organized_Data of Patients.xlsx grades DR per eye (OS/OD), not per
# patient, and corrected_manifest.csv doesn't say which eye a row is -- tianjin_dataset.py /
# module3/dataset.py both require laterality_resolved.csv to exist in TIANJIN_DIR before they
# can run. Reuse a Drive-persisted copy if one already exists (resolving ~1,100 images takes a
# few minutes); otherwise compute it fresh here and persist it to Drive for next time.
# This branch re-tuned module1/resolve_eye_laterality.py (item 7.1), so it gets its own copy
# (module1_cache<RUN_SUFFIX>/); the settings come from notebook 11's EyePACS validation when it exists.
# _v2: the first file on this branch used the EyePACS validation's convention at only ~61 % accuracy
# (EyePACS mixes mirrored and non-mirrored images) -> 20.6 % of patients had both eyes labelled alike.
# Round 5: its own copy in this round's cache folder (the text-safe crop changes the field of view the
# optic-disc position is measured in); rounds 3/4 ('' / '_r4') share module1_cache<RUN_SUFFIX>/.
_LAT_DIR = 'module1_cache' + (RUN_SUFFIX if ROUND_TAG in ('', '_r4') else RUN_TAG)
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, _LAT_DIR, 'laterality_resolved_v2.csv')
os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
LATERALITY_VALIDATION = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_eyepacs_validation.json')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    import shutil
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive: {LATERALITY_DRIVE_PATH}')
else:
    %cd {REPO_CODE_DIR}
    # LATERALITY_DISC_LEFT_IS: which eye a left-of-centre optic disc means in Tianjin's images. None =
    # the script's default ('OD', what every earlier run used), unless notebook 11's EyePACS validation
    # is clearly reliable (>= 80 % accuracy). Check a few Tianjin images of known right eyes by eye and
    # set 'OD' or 'OS' here -- that is the real answer (after changing it, delete laterality_resolved_v2.csv
    # on Drive so it is recomputed).
    LATERALITY_DISC_LEFT_IS = 'OS'  # round 3 set this by hand after checking Tianjin images by eye
    _lat_args = ''
    _b = None
    if os.path.isfile(LATERALITY_VALIDATION):
        import json as _json
        _b = _json.load(open(LATERALITY_VALIDATION))['best']
    if LATERALITY_DISC_LEFT_IS:
        _lat_args = f' --disc-left-is {LATERALITY_DISC_LEFT_IS}'
        print(f'Laterality: --disc-left-is {LATERALITY_DISC_LEFT_IS} (set by hand)')
    elif _b and _b['accuracy_on_decided'] >= 0.8:
        _lat_args = (f" --disc-left-is {_b['disc_left_is']} --margin {_b['margin']}"
                     + (' --legacy-estimator' if _b['estimator'] == 'legacy' else ''))
        print(f"Laterality settings from the EyePACS validation: {_lat_args} "
              f"(accuracy {_b['accuracy_on_decided']:.1%}, {_b['uncertain_rate']:.1%} uncertain)")
    else:
        print('\u26a0 Laterality: script defaults (--disc-left-is OD --margin 0.1) -- '
              + (f"the EyePACS validation is too weak to use ({_b['accuracy_on_decided']:.1%} accuracy). "
                 if _b else 'no EyePACS validation. ')
              + 'Set LATERALITY_DISC_LEFT_IS after checking a few Tianjin images by eye.')
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"{_lat_args}
    # Read the printed same-eye disagreement rate above before trusting this for training --
    # see resolve_eye_laterality.py's module docstring (>5% disagreement means the heuristic
    # needs tuning).
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    import shutil
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive: {LATERALITY_DRIVE_PATH}')

✓ Reused laterality_resolved.csv from Drive: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/laterality_resolved_v2.csv


In [5]:
# Sanity-check the survival dataset builds correctly against the real download before
# committing to a full training run -- prints patient counts, event rate, and LBS stratum
# thresholds per grade.
%cd {REPO_CODE_DIR}/module3
!python dataset.py --dataset-dir "{TIANJIN_DIR}" --module1-cache "{TIANJIN_CACHE}"

/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module3
Testing Tianjin Survival Dataset (Module 3)
✓ Loaded Module 1 cache: 1234 images (/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt)
  Total eye-pairs in corrected_manifest.csv: 1115
  Baseline sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', OD <- 'OD Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', worse-eye <- 'At-risk Eye Grade（Worse eye）'
  Follow-up sheet '2 year follow-up': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopath

In [6]:
# Round 4, Phase-0 check (item R4-0.3): how many eyes and progression events per baseline grade -- decides
# whether the in-scope subset (baseline >= mild NPDR) can be TRAINED on alone or only REPORTED on.
import sys, pandas as pd
sys.path.insert(0, f'{REPO_CODE_DIR}/module3'); sys.path.insert(0, REPO_CODE_DIR)
from dataset import TianjinSurvivalDataset
_m = TianjinSurvivalDataset(TIANJIN_DIR, module1_cache_path=TIANJIN_CACHE, image_size=64).manifest
_t = _m.groupby('baseline_icdr').agg(eyes=('event', 'size'), events=('event', 'sum'), patients=('patient_id', 'nunique'))
_t['event_rate'] = (_t['events'] / _t['eyes']).round(3)
display(_t)
_in = _m[_m['baseline_icdr'] >= 1]
print(f"In scope (baseline >= 1): {len(_in)} eyes, {int(_in['event'].sum())} events, {_in['patient_id'].nunique()} patients "
      f"-> ~{int(round(_in['event'].sum() * 0.15))} events in a 15 % test split.")
print('Fewer than ~40 in-scope events: keep MIN_BASELINE_GRADE = 0 (train on all, report in scope) -- the default.')

✓ Loaded Module 1 cache: 1234 images (/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt)
  Total eye-pairs in corrected_manifest.csv: 1115
  Baseline sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', OD <- 'OD Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', worse-eye <- 'At-risk Eye Grade（Worse eye）'
  Follow-up sheet '2 year follow-up': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Prol

,eyes,events,patients,event_rate
baseline_icdr,,,,
0,866,103,472,0.119
1,76,12,64,0.158
2,140,16,97,0.114
3,14,3,9,0.214
4,2,0,2,0.000


In scope (baseline >= 1): 232 eyes, 31 events, 154 patients -> ~5 events in a 15 % test split.
Fewer than ~40 in-scope events: keep MIN_BASELINE_GRADE = 0 (train on all, report in scope) -- the default.


In [7]:
# MODULE3_EPOCHS: 30 (was 10). Training state is saved every epoch on the runtime's disk and
# copied to Drive every 10 epochs and after the last one; if Colab disconnects, re-run and it
# continues from the last Drive copy (delete resume_state.pt below and restart the runtime to
# start over). Best-epoch saving: after every epoch the model is scored on the
# held-out val patients (NLL); the best epoch is saved as best-model.ckpt and is the one used
# for the metrics, the figures below and notebooks 08 / 10. final-model.ckpt is the last epoch.
# On this branch (CODE_FIXES section 6): patients are split 70 / 15 / 15 into train / val / TEST --
# val picks the epoch, the reported AUROC / Brier (poc_results.json 'metrics', and the figures) are
# on the test patients only (val numbers: 'val_metrics'); the first 6 EfficientNet-B4 blocks stay
# frozen, training images are augmented and AdamW uses weight decay 1e-2. The earlier setup:
# --test-fraction 0 --val-fraction 0.2 --freeze-blocks 0 --no-augment --weight-decay 0.
# Stage 0 baselines (item 7.2): 0 keeps every baseline grade (the default -- more data, documented as
# in scope); 1 keeps only eyes that start at >= mild NPDR (the manuscript's scope). Use the same value
# in notebooks 06 and 07.
MIN_BASELINE_GRADE = 0
# Round 4: metrics also on eyes with baseline >= REPORT_MIN_BASELINE_GRADE (1 = the manuscript's scope), with
# 95 % patient-bootstrap intervals and the tabular baselines; FREEZE_BLOCKS 8 = only EfficientNet-B4's head conv
# + the survival head train (6 in round 3, which over-fitted).
REPORT_MIN_BASELINE_GRADE = 1
FREEZE_BLOCKS = 8
MODULE3_EPOCHS = 30
MODULE3_RESUME_STATE = os.path.join(DRIVE_DATA_DIR, 'module3_runs', 'module3_runs_poc' + RUN_TAG, 'resume_state.pt')
%cd {REPO_CODE_DIR}/module3
!python train_module3_poc.py \
    --tianjin-dir "{TIANJIN_DIR}" \
    --tianjin-module1-cache "{TIANJIN_CACHE}" \
    --num-epochs {MODULE3_EPOCHS} --min-baseline-grade {MIN_BASELINE_GRADE} \
    --report-min-baseline-grade {REPORT_MIN_BASELINE_GRADE} --freeze-blocks {FREEZE_BLOCKS} \
    --resume-state "{MODULE3_RESUME_STATE}"

/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module3
Module 3 POC training (EfficientNet-B4 + Weibull survival head)
Device: cuda
tianjin_dir: /content/data/retinal-dr-longitudinal
tianjin_module1_cache_path: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt

[1/4] Loading Tianjin survival dataset...
✓ Loaded Module 1 cache: 1234 images (/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt)
  Total eye-pairs in corrected_manifest.csv: 1115
  Baseline sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', OD <- 'OD Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retino

In [8]:
import json, os

rp = f'{REPO_CODE_DIR}/module3/module3_runs_poc/poc_results.json'
print(json.dumps(json.load(open(rp)), indent=2) if os.path.isfile(rp) else f'{rp} not found -- the training cell above did not finish.')
if os.path.isfile(rp):  # round 4: the comparison the thesis needs, on the same test patients
    _r = json.load(open(rp))
    _rows = [('image model (EfficientNet-B4 + Weibull)', 'test', _r['test_metrics']),
             ('image model (EfficientNet-B4 + Weibull)', 'test_in_scope', (_r['test_metrics'] or {}).get('in_scope'))]
    _rows += [(b, s, m) for b, sets in _r.get('tabular_baselines', {}).items() for s, m in sets.items() if s.startswith('test')]
    print(f"\n{'model':<42}{'set':<15}{'n':>5}{'events':>8}{'AUROC':>8}  {'95% CI':<17}{'Brier':>8}{'const.':>8}")
    for _name, _set, _mm in _rows:
        if not _mm:
            continue
        _ci = _mm.get('auroc_95ci')
        print(f"{_name:<42}{_set:<15}{_mm['n']:>5}{_mm.get('n_events', 0):>8}{_mm['auroc']:>8.3f}  "
              f"{(f'[{_ci[0]:.3f}, {_ci[1]:.3f}]' if _ci else ''):<17}{_mm['brier_score']:>8.4f}"
              f"{_mm.get('brier_constant_rate', float('nan')):>8.4f}")

{
  "num_epochs": 30,
  "n_train": 770,
  "n_val": 167,
  "n_test": 161,
  "tianjin_module1_cache_path": "/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt",
  "best_epoch": 16,
  "best_val_nll": 0.3406285811327175,
  "best_checkpoint": "./module3_runs_poc/best-model.ckpt",
  "metrics_model": "best-model",
  "metrics_on": "test patients",
  "metrics": {
    "n": 161,
    "mean_predicted_progression_by_2yr": 0.11393559978615424,
    "actual_progression_rate": 0.12422360248447205,
    "auroc": 0.5109929078014184,
    "brier_score": 0.10922570984591425,
    "nll": 0.3783807043009293,
    "n_events": 20,
    "brier_constant_rate": 0.10879209907025192,
    "auroc_95ci": [
      0.37699181628091427,
      0.660628827941835
    ],
    "brier_95ci": [
      0.0625538037403616,
      0.16334365981409912
    ],
    "in_scope": {
      "n": 38,
      "n_events": 9,
      "mean_predicted_progression_by_2yr": 0.12711526218213534,
      "actual_

In [9]:
# Persist the Module 3 run (best-model.ckpt, final-model.ckpt + poc_results.json with the LBS thresholds) to Drive:
# the runtime disk is wiped when the session ends, and notebooks 08 / 10 load it from here.
import shutil
MODULE3_DRIVE_DIR = os.path.join(DRIVE_DATA_DIR, 'module3_runs', 'module3_runs_poc' + RUN_TAG)
shutil.copytree(f'{REPO_CODE_DIR}/module3/module3_runs_poc', MODULE3_DRIVE_DIR, dirs_exist_ok=True)
print('✓ Copied Module 3 run ->', MODULE3_DRIVE_DIR, sorted(os.listdir(MODULE3_DRIVE_DIR)))
for _f in ('best-model.ckpt', 'final-model.ckpt'):
    _a, _b = os.path.join(f'{REPO_CODE_DIR}/module3/module3_runs_poc', _f), os.path.join(MODULE3_DRIVE_DIR, _f)
    if os.path.isfile(_a):
        assert os.path.isfile(_b) and os.path.getsize(_b) == os.path.getsize(_a), (
            f'{_f} not fully saved to Drive ({MODULE3_DRIVE_DIR}) -- is your Google Drive full? Check drive.google.com/settings/storage and empty the Trash, then re-run this cell.')

✓ Copied Module 3 run -> /content/drive/MyDrive/Thesis_Datasets/module3_runs/module3_runs_poc_drgnet_comb_prefinals_r5 ['best-model.ckpt', 'final-model.ckpt', 'poc_results.json', 'resume_state.pt']


In [10]:
# Task M/O figures: calibration plot, ROC curve, NLL-vs-epoch training curve, and
# example per-patient survival curves by LBS stratum. Reruns the saved checkpoint over
# the same held-out val split (same seed, read back from poc_results.json) -- no retraining.
import os
FIG_DIR = os.path.join(DRIVE_DATA_DIR, 'module3_runs', 'figures' + RUN_TAG)
os.makedirs(FIG_DIR, exist_ok=True)

%cd {REPO_CODE_DIR}/module3
M3_CKPT = ('./module3_runs_poc/best-model.ckpt' if os.path.isfile('./module3_runs_poc/best-model.ckpt')
           else './module3_runs_poc/final-model.ckpt')  # best epoch; final-model = last epoch
!python plot_figures.py \
    --tianjin-dir "{TIANJIN_DIR}" \
    --tianjin-module1-cache "{TIANJIN_CACHE}" \
    --checkpoint {M3_CKPT} \
    --poc-results-json ./module3_runs_poc/poc_results.json \
    --out-dir {FIG_DIR}


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module3
Rebuilding the held-out split (test patients, else val) and running the checkpoint over it...
✓ Loaded Module 1 cache: 1234 images (/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt)
  Total eye-pairs in corrected_manifest.csv: 1115
  Baseline sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', OD <- 'OD Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', worse-eye <- 'At-risk Eye Grade（Worse eye）'
  Follow-up sheet '2 year follow-up': patient_id

## Caveats

- **Round 4:** report the model output as **P(progression by 2 years)**. Every follow-up is at t = 2, so the
  Weibull shape is not identifiable; survival curves at other horizons (figures) are extrapolation, not
  evidence. Report the AUROC with its 95 % interval, next to the tabular baselines on the same patients.

- Trained on Tianjin only -- FIRE and LongDRScreening have no visit-date metadata at all, so
  they can't feed a survival model regardless of architecture.
- Grading is resolved per eye, not per patient -- both eyes of a patient are kept as
  independent samples, split at the patient level so no patient's two eyes leak across
  train/val (see `module3/dataset.py::patient_level_split`). Rows whose eye couldn't be
  confidently resolved fall back to the patient-level "worse eye" summary grade for that row
  only -- check the printed eye-specific-vs-fallback counts before trusting the usable-N
  number below.
- Stage 0 (No DR) baselines are in scope by default (`MIN_BASELINE_GRADE = 0`; set it to 1 to restrict to
  eyes starting at >= mild NPDR, the manuscript's scope) -- a real data-volume win (hundreds of Tianjin
  patients start at Stage 0), but their Lesion Burden Scores cluster near zero with little
  spread, so the low/medium/high LBS stratification may not discriminate much within Stage 0
  specifically.
- The model conditions on the baseline eye's own severity grade (a learned embedding alongside
  the LBS stratum), letting it distinguish "mild->moderate" risk from "severe->PDR" risk.
- Every Tianjin patient shares the same fixed 2-year follow-up interval. The reported AUROC
  and mean-predicted-vs-actual rate are for "probability of progression by 2 years"
  specifically -- the model has no training signal at any other time horizon.
- "Progression" = any upward ICDR stage move between baseline and the 2-year follow-up for
  that eye, not necessarily progression to sight-threatening disease specifically. The
  dataset's own "Progression (1=No;2=Yes)" column is logged as a cross-check (per-patient, not
  per-eye).
- POC scale: ImageNet-pretrained EfficientNet-B4 fine-tuned (not trained from scratch), small
  post-exclusion N, patient-level train/val split -- not a claim of a clinically validated
  survival model.

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [11]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')

✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.
